### Spark Aggregate Functions

#### Funciones simples de agregación

In [0]:
%run "../includes/configuration"

In [0]:
movies_df = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
from pyspark.sql.functions import count, countDistinct, sum

In [0]:
movies_df.select(count("*")).show()

+--------+
|count(1)|
+--------+
|    4725|
+--------+



In [0]:
movies_df.select(count("year_release_date")).show()

+------------------------+
|count(year_release_date)|
+------------------------+
|                    4426|
+------------------------+



In [0]:
movies_df.select(countDistinct("year_release_date")).show()

+---------------------------------+
|count(DISTINCT year_release_date)|
+---------------------------------+
|                               90|
+---------------------------------+



In [0]:
display(movies_df.select(sum("budget")))

sum(budget)
140606455289


In [0]:
movies_df.filter("year_release_date = 2016") \
    .select(sum("budget"), count("movie_id")) \
    .withColumnRenamed("sum(budget)", "total_budget") \
    .withColumnRenamed("count(movie_id)", "count_movies") \
    .display()

total_budget,count_movies
4290150000,94


### Group By

In [0]:
from pyspark.sql.functions import count, countDistinct, sum, max, min, avg

In [0]:
movie_group_by_df = movies_df \
    .groupBy("year_release_date") \
    .agg(
        sum("budget").alias("total_budget"),
        avg("budget").alias("avg_budget"),
        max("budget").alias("max_budget"),
        min("budget").alias("min_budget"),
        count("movie_id").alias("count_movies")
    )

In [0]:
display(movie_group_by_df)

year_release_date,total_budget,avg_budget,max_budget,min_budget,count_movies
2003,5173137898,3.359180453246753E7,200000000,46000,154
2007,5711147510,3.1379931373626374E7,300000000,15000,182
2015,6508097363,3.2869178601010103E7,280000000,50000,198
2006,6398129867,2.9484469433179725E7,270000000,100000,217
2013,7962019899,3.703265069302326E7,255000000,22000,215
null,7889118615,2.6385012090301003E7,210000000,31192,299
1997,3581250000,3.6543367346938774E7,200000000,250000,98
2014,6976953000,3.114711160714286E7,250000000,300000,224
2004,5935590750,3.2084274324324325E7,200000000,65000,185
1996,2743092051,2.9816217945652176E7,100000000,200000,92


### Window Functions

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc, dense_rank

In [0]:
movie_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))
movie_dense_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))

movies_df.select("title", "budget", "year_release_date") \
    .filter("year_release_date is not null") \
    .withColumn("rank", rank().over(movie_rank)) \
    .withColumn("dense_rank", dense_rank().over(movie_dense_rank)) \
    .display()

title,budget,year_release_date,rank,dense_rank
Intolerance,385907,1916,1,1
The Big Parade,245000,1925,1,1
Metropolis,92620000,1927,1,1
Die Büchse der Pandora,2000000,1929,1,1
The Broadway Melody,379000,1929,2,2
Hell's Angels,3950000,1930,1,1
A Farewell to Arms,2000000,1932,1,1
42nd Street,439000,1933,1,1
It Happened One Night,325000,1934,1,1
Top Hat,609000,1935,1,1
